# Linear models for funding-aligned returns and direction

This execution notebook submits the complete declared linear-model population to the shared
research boundary. The request resolves the label artifact, task, continuous evaluation target,
fold geometry, estimator parameters, and exact eligible prediction keys before fitting. Model
interpretation is developed in `12_model_analysis`.

**Learning objectives**

- construct linear regression and classification requests from the published menu;
- inspect target, fold, and prediction-coverage identity before fitting; and
- verify that every declared result enters the prediction catalog with complete lineage.

**Book reference:** Chapter 11, linear models for trading signals.

**Prerequisites:** finalized crypto labels, features, and purged walk-forward folds.

In [1]:
import os

import polars as pl

from case_studies.crypto_perps_funding.research_workflow import (
    ALL_LABELS,
    declared_contracts,
    freeze_official_model_population,
    model_request_catalog,
    open_study,
    plan_model_catalog,
    run_model_plan,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE = os.environ.get("ML4T_OUTPUT_DIR", "")
LABELS = ALL_LABELS
PREVIEW_REDUCTIONS = {}
OVERRIDES = {}

## Declared requests

Every label and configured estimator is visible before execution. Preview reductions are part of
the resolved identity and cannot enter the canonical catalog.

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
official_population = (
    freeze_official_model_population(study) if EXECUTION_TIER == "canonical" else None
)
requests = model_request_catalog("linear", labels=LABELS)
requests

family,label,config_name
str,str,str
"""linear""","""fwd_ret_8h""","""ols"""
"""linear""","""fwd_ret_8h""","""ridge_a0.001"""
"""linear""","""fwd_ret_8h""","""ridge_a0.01"""
"""linear""","""fwd_ret_8h""","""ridge_a0.1"""
"""linear""","""fwd_ret_8h""","""ridge_a1.0"""
…,…,…
"""linear""","""fwd_dir_8h_3c""","""logistic_l1_C0.01"""
"""linear""","""fwd_dir_8h_3c""","""logistic_l1_C0.1"""
"""linear""","""fwd_dir_8h_3c""","""logistic_l1_C1.0"""


In [4]:
plan = plan_model_catalog(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides=OVERRIDES,
    preview_reductions=PREVIEW_REDUCTIONS,
)
resolved_contracts = declared_contracts(plan).select(
    "label",
    "config_name",
    "task",
    "continuous_eval_label",
    "eligible_rows",
    "training_hash",
)
resolved_contracts

label,config_name,task,continuous_eval_label,eligible_rows,training_hash
str,str,str,str,i64,str
"""fwd_ret_8h""","""ols""","""regression""",null,35280,"""686e3a1b6c39"""
"""fwd_ret_8h""","""ridge_a0.001""","""regression""",null,35280,"""e7fe69dd4db7"""
"""fwd_ret_8h""","""ridge_a0.01""","""regression""",null,35280,"""5a46eb6858de"""
"""fwd_ret_8h""","""ridge_a0.1""","""regression""",null,35280,"""3151f196533d"""
"""fwd_ret_8h""","""ridge_a1.0""","""regression""",null,35280,"""312532ac6fad"""
…,…,…,…,…,…
"""fwd_dir_8h_3c""","""logistic_l1_C0.01""","""classification""","""fwd_ret_8h""",35280,"""ae8dd3cc00ea"""
"""fwd_dir_8h_3c""","""logistic_l1_C0.1""","""classification""","""fwd_ret_8h""",35280,"""dc33a793cf4f"""
"""fwd_dir_8h_3c""","""logistic_l1_C1.0""","""classification""","""fwd_ret_8h""",35280,"""68a69c3ebbb0"""


The complete case-wide population is recorded before the first fit, so a member that later
fails to train cannot quietly disappear from the population it was declared in. This notebook
produces one slice of it, and that slice must lie inside the declaration.

In [5]:
if official_population is not None:
    outside = set(plan.expected_prediction_hashes) - set(official_population.members)
    if outside:
        raise RuntimeError(
            f"{len(outside)} declared checkpoints lie outside the official model population"
        )

## Execute and expose the catalog rows

The shared runner validates exact keys, fold completeness, finite predictions, and fitted-state
lineage before a row is marked complete.

In [6]:
execution = run_model_plan(
    plan,
    population_name="crypto-linear-validation-predictions-v1"
    if EXECUTION_TIER == "canonical"
    else None,
)
catalog = execution.catalog_rows.sort("label", "config_name", "checkpoint_value")
if (
    catalog.height != len(plan.expected_prediction_hashes)
    or catalog.filter(~pl.col("complete")).height
):
    raise RuntimeError("linear execution returned an incomplete catalog row")
catalog.select(
    "label",
    "config_name",
    "task",
    "checkpoint_kind",
    "checkpoint_value",
    "training_hash",
    "prediction_hash",
    "complete",
)

label,config_name,task,checkpoint_kind,checkpoint_value,training_hash,prediction_hash,complete
str,str,str,str,i64,str,str,bool
"""fwd_dir_8h""","""logistic_l1_C0.01""","""classification""","""final""",null,"""8aa52114ada6""","""7174dac58d0c""",true
"""fwd_dir_8h""","""logistic_l1_C0.1""","""classification""","""final""",null,"""17e5b0b6586b""","""ec89ed920a7f""",true
"""fwd_dir_8h""","""logistic_l1_C1.0""","""classification""","""final""",null,"""8d1bf1b43ec1""","""0e0a33860f53""",true
"""fwd_dir_8h""","""logistic_l1_C10.0""","""classification""","""final""",null,"""266ac4f0ae19""","""fa5cd63b9c03""",true
"""fwd_dir_8h""","""logistic_l1_C100.0""","""classification""","""final""",null,"""7fff4109e0db""","""a049bc73ff26""",true
…,…,…,…,…,…,…,…
"""fwd_ret_8h""","""ridge_a1000.0""","""regression""","""final""",null,"""0f9b4bd7678d""","""0ccf6e12f429""",true
"""fwd_ret_8h""","""ridge_a10000.0""","""regression""","""final""",null,"""c00a96164d32""","""b7516bcd9070""",true
"""fwd_ret_8h""","""ridge_a100000.0""","""regression""","""final""",null,"""3f6ccee2dda6""","""496ba3af702e""",true


## Key takeaways and limitations

- The request is the reproducible unit: label, task, configuration, folds, and eligible keys are
  resolved before fitting.
- Classification diagnostics retain the continuous return target used for trading evaluation.
- Linear models provide an interpretable reference class but do not represent nonlinear feature
  interactions or time-dependent hidden state.